
# EDA Visual – Cuaderno de Prácticas Integradas / Visual EDA – Integrated Practice Workbook

Este cuaderno centraliza ejercicios prácticos de EDA con NumPy, pandas, Seaborn, Matplotlib y Plotly Express.
This workbook centralizes hands‑on EDA exercises with NumPy, pandas, Seaborn, Matplotlib and Plotly Express.

Formato / Format: limpio, bilingüe (ES/EN), con celdas semi‑guiadas (# TODO) y pequeñas pistas.



## 1. Descarga automática de datasets / Automatic dataset download

Descarga y guarda los datasets en `datasets/`. Si una URL falla, se intenta con otra.
Downloads and saves datasets into `datasets/`. If a URL fails, another mirror is tried.


In [1]:

# %pip -q install pandas seaborn matplotlib plotly

import os, warnings, pathlib
import pandas as pd

warnings.filterwarnings("ignore")
pathlib.Path("datasets").mkdir(exist_ok=True)

def try_download_csv(name, urls, save_as=None):
    save_as = save_as or f"{name}.csv"
    last_err = None
    for u in urls:
        try:
            df = pd.read_csv(u)
            df.to_csv(f"datasets/{save_as}", index=False)
            print(f"✓ {save_as} — {len(df):,} rows")
            return df
        except Exception as e:
            last_err = e
    print(f"✗ {name} — failed ({last_err})")
    return None

print("Downloading datasets...")

_ = try_download_csv("spotify_2023",[
    "https://raw.githubusercontent.com/yogesh-virat/Spotify-2023/main/spotify-2023.csv",
    "https://raw.githubusercontent.com/abdullahfawzy/Spotify-Dataset-2023/main/spotify-2023.csv",
    "https://raw.githubusercontent.com/raulgomez99/spotify-eda-2023/main/spotify-2023.csv",
], save_as="spotify_2023.csv")

_ = try_download_csv("titanic",[
    "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv",
    "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv",
], save_as="titanic.csv")

_ = try_download_csv("penguins",[
    "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv",
], save_as="penguins.csv")

_ = try_download_csv("flights",[
    "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/flights.csv",
], save_as="flights.csv")

_ = try_download_csv("happiness",[
    "https://raw.githubusercontent.com/plotly/datasets/master/2019.csv",
    "https://raw.githubusercontent.com/ajaykuma/DataSets/master/WorldHappinessReport.csv",
], save_as="happiness.csv")

_ = try_download_csv("netflix_titles",[
    "https://raw.githubusercontent.com/erikgregoryweb/Netflix-Data-EDA/main/netflix_titles.csv",
    "https://raw.githubusercontent.com/vega/vega-datasets/master/data/netflix_titles.csv",
], save_as="netflix_titles.csv")

print("Done.")


✗ spotify_2023 — failed (HTTP Error 404: Not Found)
✓ titanic.csv — 891 rows
✓ penguins.csv — 344 rows
✓ flights.csv — 144 rows
✗ happiness — failed (HTTP Error 404: Not Found)
✗ netflix_titles — failed (HTTP Error 404: Not Found)
Done.



### Datasets disponibles / Available datasets


In [ ]:

import os, pandas as pd
print(os.listdir("datasets"))
# Cargar dinámicamente / Load dynamically:
# name = "spotify_2023.csv"
# df = pd.read_csv(f"datasets/{name}")
# df.head()



## 2. NumPy — Operaciones y análisis numérico / NumPy — Operations and numeric analysis

Basado en ejercicios introductorios de arrays y estadísticas.  
Goal: array creation, masks, summary stats and simple simulations.


In [ ]:

import numpy as np

# 2.1 Array 4x4 y diagonal / and main diagonal
a = np.arange(16).reshape(4,4)
print(a)
diag = np.diag(a)
print("Diagonal:", diag)

# 2.2 1000 valores ~N(0,1): media y desviación / mean & std
x = np.random.randn(1000)
print("mean=%.3f std=%.3f" % (x.mean(), x.std()))

# 2.3 Cuadrados sin bucles / squares without loops
x = np.array([1,2,3,4,5])
print("squares:", x**2)

# TODO 2.4: Simula 10,000 lanzamientos de dado y estima P(X>4).
# Hint: np.random.randint and mean on the boolean mask.



## 3. pandas — Titanic (EDA estructurado) / pandas — Titanic (structured EDA)


In [ ]:

import pandas as pd
titanic = pd.read_csv("datasets/titanic.csv")
titanic.head()


In [ ]:

titanic.info()
titanic.isna().sum()


In [ ]:

titanic.groupby(["Pclass","Sex"])["Survived"].agg(["mean","count"]).reset_index()


In [ ]:

# TODO: Crea 'IsChild' (Age<12) y calcula supervivencia por Pclass y IsChild.



## 4. Seaborn — Spotify 2023 (EDA visual)


In [ ]:

import seaborn as sns, matplotlib.pyplot as plt
sns.set_theme(style="whitegrid", palette="muted")

spotify = pd.read_csv("datasets/spotify_2023.csv")
spotify.columns = [c.strip().lower().replace(' ','_').replace('%','perc').replace('-','_') for c in spotify.columns]

c_pop   = "popularity" if "popularity" in spotify.columns else None
c_dance = "danceability_perc" if "danceability_perc" in spotify.columns else ("danceability" if "danceability" in spotify.columns else None)
c_genre = "genre" if "genre" in spotify.columns else ("track_genre" if "track_genre" in spotify.columns else None)

if c_pop:
    sns.histplot(spotify[c_pop], bins=30, kde=True)
    plt.title("Distribución de popularidad / Popularity distribution")
    plt.show()

# TODO: Popularidad media por género (top 8). Hint: value_counts(), filter, barplot(estimator='mean').


In [ ]:

if c_pop and c_dance:
    sample = spotify.sample(min(10000, len(spotify)), random_state=42)
    sns.scatterplot(data=sample, x=c_dance, y=c_pop, alpha=0.3)
    plt.title("Popularity vs Danceability (sample)")
    plt.show()

# TODO: lmplot con hue=c_genre (si pocos géneros).



## 5. Matplotlib — Penguins (personalización y exportación)


In [ ]:

import matplotlib.pyplot as plt
peng = pd.read_csv("datasets/penguins.csv").dropna()

plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(6,4))
ax.scatter(peng["flipper_length_mm"], peng["body_mass_g"], alpha=0.6, edgecolors="k", linewidths=0.2)
ax.set_title("Penguins: Flipper length vs Body mass")
ax.set_xlabel("Flipper length (mm)")
ax.set_ylabel("Body mass (g)")
ax.annotate("Annotation example",
            xy=(peng["flipper_length_mm"].median(), peng["body_mass_g"].median()),
            xytext=(190, 6000),
            arrowprops=dict(arrowstyle="->", lw=1))
plt.tight_layout()
plt.savefig("penguins_scatter.png", dpi=300, bbox_inches="tight")
plt.show()

# TODO: Cambia estilo a 'ggplot' y añade axhline con media de body_mass_g.



## 6. Plotly Express — Flights (interactivo) / interactive


In [ ]:

import plotly.express as px
flights = pd.read_csv("datasets/flights.csv")
fig = px.line(flights, x="month", y="passengers", color="year",
              title="Passengers by month and year")
fig.show()

# TODO: Heatmap de estacionalidad (pivot + px.imshow o px.density_heatmap).



## 7. Proyecto EDA libre / Open EDA project

Elige un dataset (Spotify, Titanic, Penguins, Flights, Happiness, Netflix) y realiza un EDA visual completo.
Choose one dataset and perform a complete visual EDA.

Requisitos / Requirements:
- Mínimo 4 gráficos (univariante, bivariante, categórico, multivariante).
- Conclusiones escritas (5–8 líneas).
- Guarda al menos una figura a 300 DPI.


In [ ]:

# name = "spotify_2023.csv"
# df = pd.read_csv(f"datasets/{name}")
# df.head()



## 8. Explorador de datasets (opcional) / Optional dataset explorer


In [2]:

import os, pandas as pd
available = os.listdir("datasets")
print("Available:", available)

# name = available[0]
# df = pd.read_csv(f"datasets/{name}")
# df.head()


Available: ['flights.csv', 'titanic.csv', 'penguins.csv']
